In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/nyc_sales_property_cleaned.csv")

print("Shape:", df.shape)
df.head()

Shape: (650335, 20)


,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AS OF FINAL ROLL,BLOCK,LOT,BUILDING CLASS AS OF FINAL ROLL,ADDRESS,APARTMENT NUMBER,ZIP CODE,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE
0,1,CHINATOWN,47 CONDO NON-BUSINESS STORAGE,4,172,1490,RS,"50 FRANKLIN STREET, S9B",S9B,10013.0,0.0,1.0,1.0,0.0,0.0,2007.0,4,RS,1590000,2019-06-21
1,1,FASHION,21 OFFICE BUILDINGS,4,779,23,O6,227 WEST 29TH STREET,NaN,10001.0,0.0,3.0,3.0,NaN,NaN,1927.0,4,O6,0,2019-05-03
2,1,FASHION,13 CONDOS - ELEVATOR APARTMENTS,2,841,1044,R4,"16 WEST 40TH STREET, 26A",26A,10018.0,1.0,0.0,1.0,0.0,0.0,2015.0,2,R4,5203258,2019-03-11
3,1,CHELSEA,22 STORE BUILDINGS,4,774,79,K4,276 WEST 25TH STREET,NaN,10001.0,0.0,2.0,2.0,NaN,NaN,1910.0,4,K4,6900000,2019-08-02
4,1,CHELSEA,21 OFFICE BUILDINGS,4,802,75,O6,158 WEST 27 STREET,NaN,10001.0,0.0,14.0,14.0,NaN,NaN,1913.0,4,O6,99350000,2019-10-24


In [3]:
df["SALE YEAR"] = pd.to_datetime(df["SALE DATE"]).dt.year

df["PROPERTY AGE"] = df["SALE YEAR"] - df["YEAR BUILT"]

df["PRICE PER SQ FT"] = np.where(
    (df["GROSS SQUARE FEET"] > 0) & (df["SALE PRICE"] > 0),
    df["SALE PRICE"] / df["GROSS SQUARE FEET"],
    np.nan
)

print([
    "SALE YEAR",
    "PROPERTY AGE",
    "PRICE PER SQ FT"
])

df[["SALE DATE", "SALE YEAR", "YEAR BUILT", "PROPERTY AGE", "SALE PRICE", "GROSS SQUARE FEET", "PRICE PER SQ FT"]].head()

['SALE YEAR', 'PROPERTY AGE', 'PRICE PER SQ FT']


,SALE DATE,SALE YEAR,YEAR BUILT,PROPERTY AGE,SALE PRICE,GROSS SQUARE FEET,PRICE PER SQ FT
0,2019-06-21,2019,2007.0,12.0,1590000,0.0,NaN
1,2019-05-03,2019,1927.0,92.0,0,NaN,NaN
2,2019-03-11,2019,2015.0,4.0,5203258,0.0,NaN
3,2019-08-02,2019,1910.0,109.0,6900000,NaN,NaN
4,2019-10-24,2019,1913.0,106.0,99350000,NaN,NaN


In [4]:
print("Sales years:")
print(df["SALE YEAR"].value_counts().sort_index())

print("\nBoroughs:")
print(df["BOROUGH"].value_counts().sort_index())

print("\nProperty age:")
print(df["PROPERTY AGE"].describe())

print("\nSale Price:")
print(df["SALE PRICE"].describe())

Sales years:
SALE YEAR
2016    80068
2017    79674
2018    76659
2019    61222
2020    51061
2021    69256
2022    65018
2023    53318
2024    54812
2025    59247
Name: count, dtype: int64

Boroughs:
BOROUGH
1    104926
2     55386
3    196231
4    215624
5     78168
Name: count, dtype: int64

Property age:
count    650335.000000
mean         68.446165
std          36.532838
min           0.000000
25%          39.000000
50%          77.000000
75%          97.000000
max         337.000000
Name: PROPERTY AGE, dtype: float64

Sale Price:
count    6.503350e+05
mean     1.275999e+06
std      1.110867e+07
min      0.000000e+00
25%      0.000000e+00
50%      4.929000e+05
75%      9.450000e+05
max      2.397502e+09
Name: SALE PRICE, dtype: float64


In [5]:
print("Extreme property age records:")
print(
    df.loc[
        df["PROPERTY AGE"] > 200,
        [
            "BOROUGH",
            "NEIGHBORHOOD",
            "BUILDING CLASS CATEGORY",
            "YEAR BUILT",
            "SALE YEAR",
            "PROPERTY AGE",
            "SALE PRICE"
        ]
    ].sort_values("PROPERTY AGE", ascending=False).head(20)
)

print("\nExtreme sale price records:")
print(
    df.loc[
        df["SALE PRICE"] > 100_000_000,
        [
            "BOROUGH",
            "NEIGHBORHOOD",
            "BUILDING CLASS CATEGORY",
            "YEAR BUILT",
            "SALE YEAR",
            "PROPERTY AGE",
            "SALE PRICE"
        ]
    ].sort_values("SALE PRICE", ascending=False).head(20)
)

Extreme property age records:
        BOROUGH        NEIGHBORHOOD            BUILDING CLASS CATEGORY  \
290846        5         PRINCES BAY            01 ONE FAMILY DWELLINGS   
610196        3          OCEAN HILL             14 RENTALS - 4-10 UNIT   
604021        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
615090        1        CIVIC CENTER  15 CONDOS - 2-10 UNIT RESIDENTIAL   
615091        1        CIVIC CENTER  15 CONDOS - 2-10 UNIT RESIDENTIAL   
604017        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
619733        1                SOHO             14 RENTALS - 4-10 UNIT   
604024        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604023        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604025        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604022        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604020        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604019  

In [6]:
sales_by_year=(
    df.groupby("SALE YEAR").size().reset_index(name="TRANSACTION COUNT")
)

print(sales_by_year)

   SALE YEAR  TRANSACTION COUNT
0       2016              80068
1       2017              79674
2       2018              76659
3       2019              61222
4       2020              51061
5       2021              69256
6       2022              65018
7       2023              53318
8       2024              54812
9       2025              59247


In [7]:
sales_by_borough = (
    df.groupby("BOROUGH").size().reset_index(name="TRANSACTION COUNT").sort_values("TRANSACTION COUNT", ascending=False)
)

print(sales_by_borough)

   BOROUGH  TRANSACTION COUNT
3        4             215624
2        3             196231
0        1             104926
4        5              78168
1        2              55386


In [8]:
borough_names = {
    1: "Manhattan",
    2: "Bronx",
    3: "Brooklyn",
    4: "Queens",
    5: "Staten Island"
}

df["BOROUGH NAME"] = df["BOROUGH"].map(borough_names)

print(df["BOROUGH NAME"].value_counts())

BOROUGH NAME
Queens           215624
Brooklyn         196231
Manhattan        104926
Staten Island     78168
Bronx             55386
Name: count, dtype: int64


In [9]:
sales_by_neighborhood = (
    df.groupby("NEIGHBORHOOD").size().reset_index(name="TRANSACTION COUNT").sort_values("TRANSACTION COUNT", ascending=False)
)

print(sales_by_neighborhood.head(20))

                NEIGHBORHOOD  TRANSACTION COUNT
91            FLUSHING-NORTH              22500
15        BEDFORD STUYVESANT              12348
24              BOROUGH PARK              11808
8                    ASTORIA               9920
13                   BAYSIDE               9499
235  UPPER EAST SIDE (59-79)               8737
72             EAST NEW YORK               8236
236  UPPER EAST SIDE (79-96)               7993
84             FLATBUSH-EAST               7948
198            RICHMOND HILL               7753
106              GREAT KILLS               7655
142         LONG ISLAND CITY               7339
39                  CANARSIE               7338
238  UPPER WEST SIDE (59-79)               7219
61             CROWN HEIGHTS               6906
11                 BAY RIDGE               6677
157             MIDTOWN WEST               6669
76                  ELMHURST               6654
92            FLUSHING-SOUTH               6604
213           SHEEPSHEAD BAY            

In [10]:
sales_by_property_type = (
    df.groupby("BUILDING CLASS CATEGORY").size().reset_index(name="TRANSACTION COUNT").sort_values("TRANSACTION COUNT", ascending=False)
)

print(sales_by_property_type.head(20))

              BUILDING CLASS CATEGORY  TRANSACTION COUNT
1             01 ONE FAMILY DWELLINGS             166441
3             02 TWO FAMILY DWELLINGS             140903
26    13 CONDOS - ELEVATOR APARTMENTS              92687
5           03 THREE FAMILY DWELLINGS              39686
19     10 COOPS - ELEVATOR APARTMENTS              26857
13     07 RENTALS - WALKUP APARTMENTS              26131
0            01  ONE FAMILY DWELLINGS              18440
2            02  TWO FAMILY DWELLINGS              16339
18    10  COOPS - ELEVATOR APARTMENTS              13327
7               04 TAX CLASS 1 CONDOS              11315
30  15 CONDOS - 2-10 UNIT RESIDENTIAL              11222
25   13  CONDOS - ELEVATOR APARTMENTS               9265
38                 22 STORE BUILDINGS               7834
24      12 CONDOS - WALKUP APARTMENTS               6209
17       09 COOPS - WALKUP APARTMENTS               5666
28             14 RENTALS - 4-10 UNIT               5367
4          03  THREE FAMILY DWE